# Kraftsystemmålinger (Power system measurements)

Omfatter:

- Supervisory Control and Data Acquisition (SCADA)
- Phasor Measurement Unit (PMU)
- andre sensordata

Hittil har vi sett på kraftsystemdata fra Statnett og ENTSO-E, og sett på hvordan vi kan aksessere dette både ved å laste ned .csv filer fra nettsiden og gjennom API.

Tabellen nedenfor viser hvordan dataene gjøres tilgjengelig gjennom brukergrensesnitt. 

| Del      | Hva det er              |
|----------|------------------------|
| GUI      | Grafisk burkergrensesnitt|
| Frontend | Visualisering          |
| Backend  | Målinger + databaser   |

Statnett og ENTSO-E er gode eksempler her fordi i tilfellet Statnett har en grensesnittet bestemt hvilke data vi ser og får lov til å laste ned, mens i ENTSO-E-tilfellet har vi direkte tilgang til databasen og bestemmer selv hva vi laster ned.

Som nevnt tidligere er API-nøkkelen en digital-ID som identifiserer den som får tilgang til å laste ned rådata

### Driftsdata (operasjonell data)
Fokus på drift:

SCADA-data
brukes av operatører

### Wide Area Monitoring (WAMS)
Mer spesifikt for PMU:

- synkroniserte målinger
- brukes i stabilitetsanalyse
- dekker store geografiske områder

SCADA data er operasjonell data og ofte propietær og man må ha spesiell tilgang for å aksessere denne type data. [IEEE DataPort](https://ieee-dataport.org/keywords/scada) er en online plattform der forskere og ingeniører kan dele, finne og bruke [datasett](https://ieee-dataport.org/datasets).
Det er sjeldent at man finner ekte SCADA data her av grunner nevnt ovenfor men på GitHub, en plattform for å lagre, dele og samarbeide på kode og prosjekter, er noe tilgjengelig på [OpenWindSCADA](https://github.com/sltzgs/OpenWindSCADA). Github ble laget av utviklere i 2008 og kjøpt av Microsoft i 2018. OpenWindSCADA samler lenker til andre dataplattformer som til open-source plattformen [Zenodo](https://zenodo.org/) initiert av partikkel-fysikk laben CERN i Sveits og støttet av EU. På Zenodo kan man laste ned åpen SCADA data fra [Kelmarsh windfarm](https://zenodo.org/records/5841834?preview_file=Kelmarsh_Grid_3088.zip#.YgpBQ_so-V7)

Du kan laste ned datasettet her:

- [Last ned SCADA vindturbindata.csv](../data/SCADA_Wind_turbine_data_2021.csv)

Dette datasettet har mange kolonner (300+) og en blanding av: tall, tekst, og manglende verdier

Vi får da feilmeldingen `DtypeWarning: Columns (...) have mixed types`

Det betyr at samme kolonne inneholder ulike datatyper

Vi bruker `low_memory=False`

dette forteller pandas:

“les hele filen før du bestemmer datatype”

In [2]:
import pandas as pd
data = pd.read_csv('../data/SCADA_Wind_turbine_data_2021.csv', low_memory=False)

In [3]:
data.head(3)

,# This file was exported by Greenbyte at 2022-01-27 22:46:00. Please see https://www.greenbyte.com for more information about Greenbyte.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 289,Unnamed: 290,Unnamed: 291,Unnamed: 292,Unnamed: 293,Unnamed: 294,Unnamed: 295,Unnamed: 296,Unnamed: 297,Unnamed: 298
0,#,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,# Turbine: Kelmarsh 6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,# Turbine type: Senvion MM92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Vi hopper over radene med metadata med 'skiprows=9'

In [4]:
df = pd.read_csv('../data/SCADA_Wind_turbine_data_2021.csv', skiprows=9, header=0,low_memory=False)

In [5]:
df.head(3)

,# Date and time,Wind speed (m/s),"Wind speed, Standard deviation (m/s)","Wind speed, Minimum (m/s)","Wind speed, Maximum (m/s)",Long Term Wind (m/s),Wind speed Sensor 1 (m/s),"Wind speed Sensor 1, Standard deviation (m/s)","Wind speed Sensor 1, Minimum (m/s)","Wind speed Sensor 1, Maximum (m/s)",...,Tower Acceleration y (mm/ss),"Tower Acceleration X, Min (mm/ss)","Tower Acceleration X, Max (mm/ss)","Tower Acceleration Y, Min (mm/ss)","Tower Acceleration Y, Max (mm/ss)","Drive train acceleration, Max (mm/ss)","Drive train acceleration, Min (mm/ss)","Drive train acceleration, StdDev (mm/ss)","Tower Acceleration X, StdDev (mm/ss)","Tower Acceleration Y, StdDev (mm/ss)"
0,1/1/2021 0:00,4.038019,0.700345,2.551292,5.217981,7.1,4.126690,0.563626,3.120180,5.439071,...,38.765776,29.741989,274.860626,6.783698,68.157379,3.980548,3.980548,0.0,59.474656,17.885789
1,1/1/2021 0:10,3.710743,0.583452,2.508970,5.101780,7.1,3.986455,0.567087,2.935361,4.973041,...,29.165829,25.464008,253.534164,7.835090,58.152035,4.652066,4.652066,0.0,53.743395,13.314676
2,1/1/2021 0:20,4.118079,0.713885,2.725780,5.331213,7.1,4.165041,0.582563,3.219593,5.317415,...,29.821561,22.918325,207.772492,6.318348,84.116875,5.305034,5.305034,0.0,46.453577,20.196765


In [6]:
print(df.columns)

Index(['# Date and time', 'Wind speed (m/s)',
       'Wind speed, Standard deviation (m/s)', 'Wind speed, Minimum (m/s)',
       'Wind speed, Maximum (m/s)', 'Long Term Wind (m/s)',
       'Wind speed Sensor 1 (m/s)',
       'Wind speed Sensor 1, Standard deviation (m/s)',
       'Wind speed Sensor 1, Minimum (m/s)',
       'Wind speed Sensor 1, Maximum (m/s)',
       ...
       'Tower Acceleration y (mm/ss)', 'Tower Acceleration X, Min (mm/ss)',
       'Tower Acceleration X, Max (mm/ss)',
       'Tower Acceleration Y, Min (mm/ss)',
       'Tower Acceleration Y, Max (mm/ss)',
       'Drive train acceleration, Max (mm/ss)',
       'Drive train acceleration, Min (mm/ss)',
       'Drive train acceleration, StdDev (mm/ss)',
       'Tower Acceleration X, StdDev (mm/ss)',
       'Tower Acceleration Y, StdDev (mm/ss)'],
      dtype='object', length=299)


# Phasor Measurement Units

En Phasor Measurement Unit (PMU) er et måleinstrument i kraftsystemet som måler spenning og strøm med både størrelse og fasevinkel, synkronisert i sanntid. 

En PMU er et avansert måleinstrument som måler strømnettet i sanntid med svært høy presisjon og synkronisert med GPS-klokke.

En PMU måler typisk:

- Spenningsnivå (V)
- Spenningsvinkel ($\delta$)
- Strøm (I)
- Strømfase ($\theta$)
- Frekvens (Hz)
- Frekvensforandring ($\frac{df}{dt}$ RoCoF -rate of change of frequency)

Målinger over hele Europa skjer på akkurat samme tidspunkt med nøyaktighet på mikrosekund-nivå.

Synkroniserte målinger gjør det mulig å:

- måle faseforskjeller mellom områder
- forstå hvordan effekt flyter i nettet
- oppdage forstyrrelser umiddelbart


| Sted    | Fasevinkel |
|---------|-----------|
| Oslo    | 10°       |
| Berlin  | 5°        |

Forskjell = 5°
→ betyr at effekt flyter fra Oslo til Berlin


In [7]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

token = os.getenv("GRIDRADAR_TOKEN")

url = "https://api.gridradar.net/query"

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

data = {
    "metric": "frequency-ucte-median-1s",
    "func": "min",
    "aggr": "1m"
}

response = requests.post(url, headers=headers, json=data)

print("Status:", response.status_code)
print(response.json())


Status: 200
[{'datapoints': [[49.95759, '2026-06-09 16:42:00'], [49.952818, '2026-06-09 16:43:00'], [49.95721, '2026-06-09 16:44:00'], [49.98867, '2026-06-09 16:45:00'], [50.00178, '2026-06-09 16:46:00'], [49.993725, '2026-06-09 16:47:00'], [49.99379, '2026-06-09 16:48:00'], [49.98504, '2026-06-09 16:49:00'], [49.970435, '2026-06-09 16:50:00'], [49.97246, '2026-06-09 16:51:00'], [49.9509, '2026-06-09 16:52:00'], [49.94703, '2026-06-09 16:53:00'], [49.956685, '2026-06-09 16:54:00'], [49.958095, '2026-06-09 16:55:00'], [49.95816, '2026-06-09 16:56:00'], [49.95192, '2026-06-09 16:57:00'], [49.96601, '2026-06-09 16:58:00'], [49.96278, '2026-06-09 16:59:00'], [49.986795, '2026-06-09 17:00:00'], [50.02179, '2026-06-09 17:01:00'], [50.020675, '2026-06-09 17:02:00'], [50.01619, '2026-06-09 17:03:00'], [49.98916, '2026-06-09 17:04:00'], [49.97511, '2026-06-09 17:05:00'], [49.98083, '2026-06-09 17:06:00'], [49.969615, '2026-06-09 17:07:00'], [49.96695, '2026-06-09 17:08:00'], [49.963205, '2026-0

In [8]:
data = response.json()

# extract datapoints correctly
values = data[0]["datapoints"]

df = pd.DataFrame(values, columns=["frequency", "time"])
df["time"] = pd.to_datetime(df["time"])

df = df[["time", "frequency"]]

print(df)



                  time  frequency
0  2026-06-09 16:42:00  49.957590
1  2026-06-09 16:43:00  49.952818
2  2026-06-09 16:44:00  49.957210
3  2026-06-09 16:45:00  49.988670
4  2026-06-09 16:46:00  50.001780
5  2026-06-09 16:47:00  49.993725
6  2026-06-09 16:48:00  49.993790
7  2026-06-09 16:49:00  49.985040
8  2026-06-09 16:50:00  49.970435
9  2026-06-09 16:51:00  49.972460
10 2026-06-09 16:52:00  49.950900
11 2026-06-09 16:53:00  49.947030
12 2026-06-09 16:54:00  49.956685
13 2026-06-09 16:55:00  49.958095
14 2026-06-09 16:56:00  49.958160
15 2026-06-09 16:57:00  49.951920
16 2026-06-09 16:58:00  49.966010
17 2026-06-09 16:59:00  49.962780
18 2026-06-09 17:00:00  49.986795
19 2026-06-09 17:01:00  50.021790
20 2026-06-09 17:02:00  50.020675
21 2026-06-09 17:03:00  50.016190
22 2026-06-09 17:04:00  49.989160
23 2026-06-09 17:05:00  49.975110
24 2026-06-09 17:06:00  49.980830
25 2026-06-09 17:07:00  49.969615
26 2026-06-09 17:08:00  49.966950
27 2026-06-09 17:09:00  49.963205
28 2026-06-09 